============================================
 
Kitsap Transit — StreetLight Data 
============================================
 
CORRIDOR DESTINATION ANALYSIS

Commercial zones configured as DESTINATIONS INCLUDING EQUITY VARS.
Trip length reflects distance traveled TO each node.
Output:
  - Volume heatmaps by corridor, hour, and day type
  - Trip origin distance distributions
  - Speed patterns as congestion proxy
  - Corridor typology classification
  - Service hour gap inventory
  - Equity analysis (disability, zero-vehicle, demographics)

NOTE: Existing vehicle movement from StreetLight estimated trip counts/modeled not actual data.
 
 
=============================================================================

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import FuncFormatter
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

=============================================================================
FILE PATHS
=============================================================================

INPUT_FILE     = r'C:\Users\rebeccasc\Documents\Scripts\BI_commerce_TDM\2014361_BI_Lynwood_dest2025winterexcld0425\2014361_BI_Lynwood_dest2025winterexcld0425_od_trip_all.csv'
EQUITY_FILE    = r'C:\Users\rebeccasc\Documents\Scripts\BI_commerce_TDM\2014361_BI_Lynwood_dest2025winterexcld0425\2014361_BI_Lynwood_dest2025winterexcld0425_od_traveler_equity_all.csv'
HOUSEHOLD_FILE = r'C:\Users\rebeccasc\Documents\Scripts\BI_commerce_TDM\2014361_BI_Lynwood_dest2025winterexcld0425\2014361_BI_Lynwood_dest2025winterexcld0425_od_traveler_household_all.csv'

In [ ]:
INPUT_FILE     = r'C:\Users\rebeccasc\Documents\Scripts\BI_commerce_TDM\2014360_BI_Lynwood_dest_2024summer\2014360_BI_Lynwood_dest_2024summer_od_trip_all.csv'
EQUITY_FILE    = r'C:\Users\rebeccasc\Documents\Scripts\BI_commerce_TDM\2014360_BI_Lynwood_dest_2024summer\2014360_BI_Lynwood_dest_2024summer_od_traveler_equity_all.csv'
HOUSEHOLD_FILE = r'C:\Users\rebeccasc\Documents\Scripts\BI_commerce_TDM\2014360_BI_Lynwood_dest_2024summer\2014360_BI_Lynwood_dest_2024summer_od_traveler_household_all.csv'

In [ ]:
OUTPUT_DIR = r'C:\Users\rebeccasc\Documents\Scripts\BI_commerce_TDM\commercial_d_summer2024'
#OUTPUT_DIR = r'C:\Users\rebeccasc\Documents\Scripts\BI_commerce_TDM\commercial_d_winter2025'

In [ ]:
for subdir in ['heatmaps', 'summary_tables', 'trip_length_analysis',
               'speed_analysis', 'corridor_typology', 'gap_inventory', 'equity_analysis']:
    os.makedirs(os.path.join(OUTPUT_DIR, subdir), exist_ok=True)

=============================================================================
CORRIDOR DEFINITIONS
=============================================================================

In [ ]:
COMMERCIAL_CORRIDORS = [
    'HIGH_SCHOOL_RD',
    'SPORTSMAN_CLUB',
    'SR305_MID',
    'SR305_N',
    'SR305_S',
    'LYNWOODCENTER',
    'WINSLOW_WAY'
]

In [ ]:
CORRIDOR_DESCRIPTIONS = {
    'WINSLOW_WAY':    'Winslow Way E — downtown commercial district, ferry access approach',
    'SR305_S':        'SR 305 South — ferry terminal approach to High School Road',
    'HIGH_SCHOOL_RD': 'High School Road NE — secondary arterial, Coppertop area access',
    'SPORTSMAN_CLUB': 'Sportsman Club Road NE — secondary arterial, Coppertop node',
    'SR305_MID':      'SR 305 Mid — High School Road to Sportsman Club Road',
    'SR305_N':        'SR 305 North — Sportsman Club Road across Agate Pass',
    'LYNWOODCENTER':  'Lynwood Center Road NE — south island commercial node'
}

=============================================================================
TRIP LENGTH BINS (miles)
Zones are destinations — length = distance traveled TO the node
=============================================================================

In [ ]:
TRIP_LENGTH_BINS = {
    'Very Short (0–2 mi)': (0, 2),
    'Short (2–4 mi)':      (2, 4),
    'Medium (4–7 mi)':     (4, 7),
    'Long (7–15 mi)':      (7, 15),
    'Regional (15+ mi)':   (15, float('inf'))
}

=============================================================================
SPEED BINS (mph)
=============================================================================

SPEED_BINS = {
   'Congested (<15 mph)':  (0, 15),
   'Slow (15–25 mph)':     (15, 25),
   'Moderate (25–35 mph)': (25, 35),
   'Free Flow (35+ mph)':  (35, float('inf'))
}

=============================================================================
TIME PERIOD DEFINITIONS
Individual hour granularity — maps to StreetLight hour strings
=============================================================================

In [ ]:
# Group individual hours into meaningful display periods
HOUR_TO_PERIOD = {
     0: 'Midnight (12a–1a)',
     1: 'Late Night (1a–4a)',
     2: 'Late Night (1a–4a)',
     3: 'Late Night (1a–4a)',
     4: 'Early AM (4a–7a)',
     5: 'Early AM (4a–7a)',
     6: 'Early AM (4a–7a)',
     7: 'Morning (7a–10a)',
     8: 'Morning (7a–10a)',
     9: 'Morning (7a–10a)',
    10: 'Late Morning (10a–12p)',
    11: 'Late Morning (10a–12p)',
    12: 'Midday (12p–3p)',
    13: 'Midday (12p–3p)',
    14: 'Midday (12p–3p)',
    15: 'Early Evening (3p–6p)',
    16: 'Early Evening (3p–6p)',
    17: 'Early Evening (3p–6p)',
    18: 'Late Evening (6p–9p)',
    19: 'Late Evening (6p–9p)',
    20: 'Late Evening (6p–9p)',
    21: 'Night (9p–12a)',
    22: 'Night (9p–12a)',
    23: 'Night (9p–12a)'
}

In [ ]:
PERIOD_ORDER = [
    'Early AM (4a–7a)',
    'Morning (7a–10a)',
    'Late Morning (10a–12p)',
    'Midday (12p–3p)',
    'Early Evening (3p–6p)',
    'Late Evening (6p–9p)',
    'Night (9p–12a)',
    'Midnight (12a–1a)',
    'Late Night (1a–4a)'

In [ ]:
]

In [ ]:
# Transit service hours by day type
# Verify against current BI Ride and fixed-route schedules at kitsaptransit.com
TRANSIT_HOURS = {
    'monday':    {'start': 4.5, 'end': 19.25},
    'tuesday':   {'start': 4.5, 'end': 19.25},
    'wednesday': {'start': 4.5, 'end': 19.25},
    'thursday':  {'start': 4.5, 'end': 19.25},
    'friday':    {'start': 4.5, 'end': 19.25},
    'saturday':  {'start': 9.0, 'end': 18.0},
    'sunday':    {'start': 9.0, 'end': 16.0}
}

=============================================================================
COLUMN NAMES
=============================================================================

In [ ]:
VOLUME_COL      = 'Average Daily O-D Traffic (StL Volume)'
TRIP_LENGTH_COL = 'Avg All Trip Length (mi)'
# TRIP_SPEED_COL  = 'Avg Trip Speed (mph)'
CORRIDOR_COL    = 'Destination Zone Name'

In [ ]:
# Equity file uses different volume and corridor column names
EQ_VOL_COL = 'Average Daily O-D Traffic (StL Volume)'
EQ_CORRIDOR_COL = 'Destination Zone Name'

In [ ]:
DAY_TYPE_ORDER = [
    '0: All Days (M-Su)',
    '1: Monday (M-M)',
    '2: Tuesday (Tu-Tu)',
    '3: Wednesday (W-W)',
    '4: Thursday (Th-Th)',
    '5: Friday (F-F)',
    '6: Saturday (Sa-Sa)',
    '7: Sunday (Su-Su)'
]

In [ ]:
DAY_LABELS = {
    '0: All Days (M-Su)':   'All Days',
    '1: Monday (M-M)':      'Mon',
    '2: Tuesday (Tu-Tu)':   'Tue',
    '3: Wednesday (W-W)':   'Wed',
    '4: Thursday (Th-Th)':  'Thu',
    '5: Friday (F-F)':      'Fri',
    '6: Saturday (Sa-Sa)':  'Sat',
    '7: Sunday (Su-Su)':    'Sun'
}

In [ ]:
comma_fmt = FuncFormatter(lambda x, _: f'{x:,.0f}')

=============================================================================
HELPER FUNCTIONS
=============================================================================

In [ ]:
def extract_hour(day_part_str):
    """Extract integer hour from StreetLight hour string e.g. '13: 12pm (12noon-1pm)' -> 12"""
    try:
        s = str(day_part_str)
        # All Day record
        if 'all day' in s.lower():
            return None
        # Format: 'N: Hpm (Hpm-Hpm)' — take the number before the colon
        idx = int(s.split(':')[0].strip())
        # StreetLight uses 0-based hour index offset by some amount — map directly
        # Hour index in Day Part string corresponds to hour of day (0=midnight, 13=noon, etc.)
        # Based on sample data: '13: 12pm (12noon-1pm)' = noon = hour 12
        # So hour = index - 1
        return idx - 1
    except:
        return None

In [ ]:
def get_day_category(day_type_str):
    s = str(day_type_str).lower()
    if 'monday'    in s: return 'monday'
    if 'tuesday'   in s: return 'tuesday'
    if 'wednesday' in s: return 'wednesday'
    if 'thursday'  in s: return 'thursday'
    if 'friday'    in s: return 'friday'
    if 'saturday'  in s: return 'saturday'
    if 'sunday'    in s: return 'sunday'
    return 'all'

In [ ]:
def is_within_transit(hour, day_category):
    """Return True if hour falls within transit service hours for that day type."""
    if day_category not in TRANSIT_HOURS or hour is None:
        return None
    svc = TRANSIT_HOURS[day_category]
    return svc['start'] <= hour < svc['end']

In [ ]:
def categorize_trip_length(length):
    if pd.isna(length):
        return 'Unknown'
    for label, (lo, hi) in TRIP_LENGTH_BINS.items():
        if lo <= length < hi:
            return label
    return 'Unknown'

def categorize_speed(speed):
   if pd.isna(speed):
       return 'Unknown'
   for label, (lo, hi) in SPEED_BINS.items():
       if lo <= speed < hi:
           return label
   return 'Unknown'

In [ ]:
def weighted_avg(values, weights):
    mask = ~(pd.isna(values) | pd.isna(weights)) & (weights > 0)
    if mask.sum() == 0:
        return np.nan
    return np.average(values[mask], weights=weights[mask])

In [ ]:
def make_heatmap(pivot, title, cbar_label, cmap, vmin, vmax, fmt, outpath,
                 transit_cols=None, transit_hours=None, figsize=(14, 8)):
    """Shared heatmap renderer with optional transit hour overlay."""
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(pivot, annot=True, fmt=fmt, cmap=cmap,
                linewidths=0.5, vmin=vmin, vmax=max(pivot.values.max(), vmax * 0.01),
                cbar_kws={'label': cbar_label, 'format': comma_fmt
                          if '%' not in cbar_label else None},
                ax=ax)
    if transit_cols and transit_hours:
        row_labels = list(pivot.index)
        for col_idx, col_label in enumerate(pivot.columns):
            day_cat = transit_cols.get(col_label)
            if not day_cat or day_cat not in transit_hours:
                continue
            svc = transit_hours[day_cat]
            covered = []
            for row_idx, period in enumerate(row_labels):
                # Find hours in this period
                period_hours = [h for h, p in HOUR_TO_PERIOD.items() if p == period]
                if any(svc['start'] <= h < svc['end'] for h in period_hours):
                    covered.append(row_idx)
            if covered:
                rect = mpatches.FancyBboxPatch(
                    (col_idx, min(covered)), 1, max(covered) - min(covered) + 1,
                    boxstyle="square,pad=0", linewidth=2.5,
                    edgecolor='#00AA44', facecolor='none', linestyle='--', zorder=5)
                ax.add_patch(rect)
    ax.set_title(title, fontsize=11, pad=12)
    plt.tight_layout()
    plt.savefig(outpath, dpi=300, bbox_inches='tight')
    plt.close()

=============================================================================
LOAD & VALIDATE MAIN DATA
=============================================================================

In [ ]:
print("=" * 70)
print("BAINBRIDGE ISLAND COMMERCIAL CORRIDOR ANALYSIS")
print("Kitsap Transit — StreetLight Data, 2024")
print("=" * 70)
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

In [ ]:
try:
    df = pd.read_csv(INPUT_FILE)
    print(f"Main file: {len(df):,} records loaded.")
except Exception as e:
    print(f"ERROR reading main file: {e}")
    exit(1)

In [ ]:
# Validate corridors
found_corridors   = df[CORRIDOR_COL].unique().tolist()
missing_corridors = [c for c in COMMERCIAL_CORRIDORS if c not in found_corridors]
present_corridors = [c for c in COMMERCIAL_CORRIDORS if c in found_corridors]

In [ ]:
if missing_corridors:
    print(f"\nWARNING — corridors defined but not in data: {', '.join(missing_corridors)}")
    print("  These will be skipped. Add data and rerun to include them.\n")
print(f"Corridors present ({len(present_corridors)}/{len(COMMERCIAL_CORRIDORS)}): "
      f"{', '.join(present_corridors)}\n")

In [ ]:
# Derived columns
df['Hour']           = df['Day Part'].apply(extract_hour)
df['Period']         = df['Hour'].map(HOUR_TO_PERIOD)
df['Day_Category']   = df['Day Type'].apply(get_day_category)
df['Within_Transit'] = df.apply(
    lambda r: is_within_transit(r['Hour'], r['Day_Category']), axis=1)
df['Trip_Length_Cat'] = df[TRIP_LENGTH_COL].apply(categorize_trip_length)
# df['Speed_Cat']       = df[TRIP_SPEED_COL].apply(categorize_speed)
df['Day_Label']       = df['Day Type'].map(DAY_LABELS).fillna(df['Day Type'])

In [ ]:
# Split all-day vs hourly
df_hourly = df[df['Hour'].notna()].copy()
df_daily  = df[df['Hour'].isna()].copy()

In [ ]:
print(f"Records — hourly: {len(df_hourly):,}  |  daily totals: {len(df_daily):,}\n")

=============================================================================
LOAD EQUITY & HOUSEHOLD DATA
=============================================================================

In [ ]:
try:
    df_equity = pd.read_csv(EQUITY_FILE)
    print(f"Equity file: {len(df_equity):,} records loaded.")
except Exception as e:
    print(f"WARNING: Could not load equity file: {e}")
    df_equity = None

In [ ]:
try:
    df_household = pd.read_csv(HOUSEHOLD_FILE)
    print(f"Household file: {len(df_household):,} records loaded.")
except Exception as e:
    print(f"WARNING: Could not load household file: {e}")
    df_household = None

In [ ]:
# Convert proportions to estimated trip counts
EQUITY_COLS = [
    'With a disability', 'Without a disability',
    'White', 'Black', 'Hispanic', 'Non-Hispanic',
    'Foreign Born', "Speak English less than 'very well'"
]
HOUSEHOLD_COLS = [
    'No vehicle available', '1 vehicle available',
    '2 vehicles available', '3 or more vehicles available',
    'Owner occupied', 'Renter occupied'
]

In [ ]:
if df_equity is not None:
    df_equity['Hour']         = df_equity['Day Part'].apply(extract_hour)
    df_equity['Period']       = df_equity['Hour'].map(HOUR_TO_PERIOD)
    df_equity['Day_Category'] = df_equity['Day Type'].apply(get_day_category)
    df_equity['Day_Label']    = df_equity['Day Type'].map(DAY_LABELS).fillna(df_equity['Day Type'])
    df_equity['Within_Transit'] = df_equity.apply(
        lambda r: is_within_transit(r['Hour'], r['Day_Category']), axis=1)
    for col in EQUITY_COLS:
        if col in df_equity.columns:
            df_equity[col + '_trips'] = df_equity[col] * df_equity[EQ_VOL_COL]

In [ ]:
if df_household is not None:
    df_household['Hour']         = df_household['Day Part'].apply(extract_hour)
    df_household['Period']       = df_household['Hour'].map(HOUR_TO_PERIOD)
    df_household['Day_Category'] = df_household['Day Type'].apply(get_day_category)
    df_household['Day_Label']    = df_household['Day Type'].map(DAY_LABELS).fillna(df_household['Day Type'])
    for col in HOUSEHOLD_COLS:
        if col in df_household.columns:
            df_household[col + '_trips'] = df_household[col] * df_household[EQ_VOL_COL]

In [ ]:
print()

=============================================================================
TRANSIT OVERLAY (day label -> transit config key)
=============================================================================

In [ ]:
TRANSIT_COL_MAP = {
    'All Days': 'monday',
    'Mon': 'monday', 'Tue': 'tuesday', 'Wed': 'wednesday',
    'Thu': 'thursday', 'Fri': 'friday',
    'Sat': 'saturday', 'Sun': 'sunday'
}

=============================================================================
ANALYSIS 1: VOLUME HEATMAPS
=============================================================================

In [ ]:
print("=" * 70)
print("ANALYSIS 1: VOLUME HEATMAPS BY CORRIDOR")
print("=" * 70)

In [ ]:
day_col_order = [DAY_LABELS[d] for d in DAY_TYPE_ORDER if d in DAY_LABELS]

In [ ]:
for corridor in present_corridors:
    cdata = df_hourly[df_hourly[CORRIDOR_COL] == corridor]
    if cdata.empty:
        print(f"  {corridor}: no hourly data — skipped")
        continue

    pivot = cdata.pivot_table(
        values=VOLUME_COL,
        index='Period',
        columns='Day_Label',
        aggfunc='sum',
        fill_value=0
    )
    pivot = pivot.reindex([p for p in PERIOD_ORDER if p in pivot.index])
    pivot = pivot[[c for c in day_col_order if c in pivot.columns]]

    max_val = pivot.values.max()
    desc    = CORRIDOR_DESCRIPTIONS.get(corridor, corridor)

    make_heatmap(
        pivot=pivot,
        title=(f'Arriving Vehicle Trips — {corridor}\n{desc}\n'
               f'Green dashed outline = current Kitsap Transit service hours'),
        cbar_label='Avg Daily Vehicle Trips',
        cmap='YlOrRd', vmin=0, vmax=max_val, fmt='.0f',
        outpath=os.path.join(OUTPUT_DIR, 'heatmaps', f'Volume_{corridor}.pdf'),
        transit_cols=TRANSIT_COL_MAP,
        transit_hours=TRANSIT_HOURS
    )
    print(f"  {corridor}: saved (peak {max_val:,.0f} trips)")

=============================================================================
ANALYSIS 2: TRIP ORIGIN DISTANCE DISTRIBUTION
=============================================================================

In [ ]:
print("\n" + "=" * 70)
print("ANALYSIS 2: TRIP ORIGIN DISTANCE DISTRIBUTION")
print("=" * 70)
print("  Zones are destinations. Trip length = distance traveled TO the node.")
print("  Nearby-origin (<4 mi): adjacent neighborhood visitors.")
print("  Distant-origin (>7 mi): likely cross-island or off-island.\n")

In [ ]:
length_cat_order       = list(TRIP_LENGTH_BINS.keys())
corridor_length_summary = []

In [ ]:
nrows = int(np.ceil(len(present_corridors) / 2))
fig, axes = plt.subplots(nrows=nrows, ncols=2, figsize=(14, 4 * nrows))
axes = axes.flatten()

In [ ]:
for idx, corridor in enumerate(present_corridors):
    cdata = df[df[CORRIDOR_COL] == corridor]
    if cdata.empty:
        continue

    length_vols = cdata.groupby('Trip_Length_Cat')[VOLUME_COL].sum()
    length_vols = length_vols.reindex(
        [c for c in length_cat_order if c in length_vols.index], fill_value=0)
    total = length_vols.sum()
    pct   = (length_vols / total * 100) if total > 0 else length_vols * 0

    nearby_pct  = pct.get('Very Short (0–2 mi)', 0) + pct.get('Short (2–4 mi)', 0)
    distant_pct = pct.get('Long (7–15 mi)', 0)      + pct.get('Regional (15+ mi)', 0)

    corridor_length_summary.append({
        'Corridor':           corridor,
        'Total_Volume':       round(total),
        'Pct_Very_Short':     round(pct.get('Very Short (0–2 mi)', 0), 1),
        'Pct_Short':          round(pct.get('Short (2–4 mi)', 0), 1),
        'Pct_Medium':         round(pct.get('Medium (4–7 mi)', 0), 1),
        'Pct_Long':           round(pct.get('Long (7–15 mi)', 0), 1),
        'Pct_Regional':       round(pct.get('Regional (15+ mi)', 0), 1),
        'Pct_Nearby_Total':   round(nearby_pct, 1),
        'Pct_Distant_Total':  round(distant_pct, 1)
    })

    ax     = axes[idx]
    colors = ['#2ecc71', '#27ae60', '#f39c12', '#e74c3c', '#922b21']
    bars   = ax.bar(range(len(pct)), pct.values, color=colors[:len(pct)])
    ax.set_xticks(range(len(pct)))
    ax.set_xticklabels(pct.index, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('% of Arriving Trips')
    ax.set_title(f'{corridor}\nTotal arriving: {total:,.0f} trips', fontsize=9)
    ax.set_ylim(0, 100)
    for bar, val in zip(bars, pct.values):
        if val > 3:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                    f'{val:.0f}%', ha='center', va='bottom', fontsize=8)

In [ ]:
for idx in range(len(present_corridors), len(axes)):
    axes[idx].set_visible(False)

In [ ]:
plt.suptitle(
    'Trip Origin Distance by Corridor — How Far Are People Traveling to Reach Each Node?\n'
    'Green = nearby origin (<4 mi)   Red = distant origin (7+ mi)',
    fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'trip_length_analysis', 'origin_distance_by_corridor.pdf'),
            dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
length_df = pd.DataFrame(corridor_length_summary)
length_df.to_csv(os.path.join(OUTPUT_DIR, 'trip_length_analysis', 'origin_distance_summary.csv'),
                 index=False)

In [ ]:
print("Origin Distance Summary:")
print(length_df[['Corridor', 'Pct_Nearby_Total', 'Pct_Distant_Total', 'Total_Volume']].to_string(index=False))

=============================================================================
ANALYSIS 3: SPEED HEATMAPS
=============================================================================

print("\n" + "=" * 70)
print("ANALYSIS 3: SPEED PATTERNS")
print("=" * 70)
print("  Speed reflects road geometry and land use as well as congestion.")
print("  Commercial nodes will show lower speeds than SR 305 segments by design.\n")

In [ ]:
# for corridor in present_corridors:
    # cdata = df_hourly[df_hourly[CORRIDOR_COL] == corridor]
    # if cdata.empty:
        # continue

    # speed_pivot = cdata.pivot_table(
      #   values=TRIP_SPEED_COL,
      #   index='Period',
      #   columns='Day_Label',
      #   aggfunc='mean'
  #   )
   #  speed_pivot = speed_pivot.reindex([p for p in PERIOD_ORDER if p in speed_pivot.index])
   #  speed_pivot = speed_pivot[[c for c in day_col_order if c in speed_pivot.columns]]

   #  make_heatmap(
   #      pivot=speed_pivot,
   #      title=(f'Average Trip Speed — {corridor}\n'
   #             f'{CORRIDOR_DESCRIPTIONS.get(corridor, "")}\n'
   #             f'Green = faster   Red = slower (reflects geometry as well as congestion)'),
   #      cbar_label='Avg Trip Speed (mph)',
   #      cmap='RdYlGn', vmin=10, vmax=45, fmt='.1f',
   #      outpath=os.path.join(OUTPUT_DIR, 'speed_analysis', f'Speed_{corridor}.pdf')
   #  )
   #  print(f"  {corridor}: speed heatmap saved")

=============================================================================
ANALYSIS 4: CORRIDOR TYPOLOGY
=============================================================================

In [ ]:
print("\n" + "=" * 70)
print("ANALYSIS 4: CORRIDOR TYPOLOGY")
print("=" * 70)

In [ ]:
typology_rows = []

In [ ]:
for corridor in present_corridors:
    cdata = df[df[CORRIDOR_COL] == corridor]
    if cdata.empty:
        continue

    total_vol   = cdata[VOLUME_COL].sum()
    avg_len     = weighted_avg(cdata[TRIP_LENGTH_COL], cdata[VOLUME_COL])
   # avg_spd     = weighted_avg(cdata[TRIP_SPEED_COL],  cdata[VOLUME_COL])

    nearby_vol  = cdata[cdata['Trip_Length_Cat'].isin(
        ['Very Short (0–2 mi)', 'Short (2–4 mi)'])][VOLUME_COL].sum()
    distant_vol = cdata[cdata['Trip_Length_Cat'].isin(
        ['Long (7–15 mi)', 'Regional (15+ mi)'])][VOLUME_COL].sum()

    pct_nearby  = (nearby_vol  / total_vol * 100) if total_vol > 0 else 0
    pct_distant = (distant_vol / total_vol * 100) if total_vol > 0 else 0

    outside_vol = cdata[cdata['Within_Transit'] == False][VOLUME_COL].sum()
    pct_outside = (outside_vol / total_vol * 100) if total_vol > 0 else 0

    if pct_nearby >= 55:
        typology      = 'Neighborhood-Serving'
        typology_note = 'Majority of arriving trips originate nearby — consistent with local commercial node access'
    elif pct_distant >= 55:
        typology      = 'Regional Draw'
        typology_note = 'Majority of arriving trips originate far away — draws cross-island or off-island visitors'
    else:
        typology      = 'Mixed'
        typology_note = 'Neither nearby nor distant origin trips dominate'

    typology_rows.append({
        'Corridor':               corridor,
        'Description':            CORRIDOR_DESCRIPTIONS.get(corridor, ''),
        'Total_Volume':           round(total_vol),
        'Avg_Origin_Distance_mi': round(avg_len, 2) if not np.isnan(avg_len) else 'N/A',
       # 'Avg_Speed_mph':          round(avg_spd, 1) if not np.isnan(avg_spd) else 'N/A',
        'Pct_Nearby_Trips':       round(pct_nearby, 1),
        'Pct_Distant_Trips':      round(pct_distant, 1),
        'Pct_Outside_Svc_Hrs':    round(pct_outside, 1),
        'Typology':               typology,
        'Typology_Note':          typology_note
    })

In [ ]:
typology_df = pd.DataFrame(typology_rows)
typology_df.to_csv(os.path.join(OUTPUT_DIR, 'corridor_typology', 'corridor_typology.csv'), index=False)

In [ ]:
print("\nCorridor Typology:")
print(typology_df[['Corridor', 'Total_Volume', 'Avg_Origin_Distance_mi',
                    'Pct_Nearby_Trips', 'Pct_Distant_Trips', 'Typology']].to_string(index=False))

In [ ]:
# Typology bar chart
fig, ax = plt.subplots(figsize=(12, 6))
x  = np.arange(len(typology_df))
w  = 0.35
b1 = ax.bar(x - w/2, typology_df['Pct_Nearby_Trips'],  w,
            label='Nearby Origin (<4 mi)', color='#2ecc71')
b2 = ax.bar(x + w/2, typology_df['Pct_Distant_Trips'], w,
            label='Distant Origin (>7 mi)', color='#e74c3c')
ax.set_xticks(x)
ax.set_xticklabels(typology_df['Corridor'], rotation=20, ha='right')
ax.set_ylabel('% of Arriving Trips')
ax.set_title('Corridor Typology — Nearby vs. Distant Origin Trip Share', fontsize=12)
ax.legend()
ax.set_ylim(0, 100)
for bar in list(b1) + list(b2):
    h = bar.get_height()
    if h > 3:
        ax.text(bar.get_x() + bar.get_width() / 2, h + 1,
                f'{h:.0f}%', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'corridor_typology', 'typology_chart.pdf'),
            dpi=300, bbox_inches='tight')
plt.close()

=============================================================================
ANALYSIS 5: SERVICE HOUR GAP INVENTORY
=============================================================================

In [ ]:
print("\n" + "=" * 70)
print("ANALYSIS 5: SERVICE HOUR GAP INVENTORY")
print("=" * 70)
print("  Identifies when vehicles arrive at commercial nodes outside KT service hours.")
print("  This is a volume observation only — not a demand confirmation.\n")

In [ ]:
gap_rows = []

In [ ]:
for corridor in present_corridors:
    cdata_h = df_hourly[df_hourly[CORRIDOR_COL] == corridor]
    if cdata_h.empty:
        continue

    total_daily = df_daily[df_daily[CORRIDOR_COL] == corridor][VOLUME_COL].sum()

    for day_label in [v for k, v in DAY_LABELS.items() if k != '0: All Days (M-Su)']:
        day_data = cdata_h[cdata_h['Day_Label'] == day_label]
        if day_data.empty:
            continue

        for period in PERIOD_ORDER:
            period_data = day_data[day_data['Period'] == period]
            if period_data.empty:
                continue

            vol          = period_data[VOLUME_COL].sum()
            in_transit   = period_data['Within_Transit'].dropna()
            coverage     = ('Within Service Hours' if in_transit.all() else
                            'Outside Service Hours' if (~in_transit).all() else
                            'Partial Overlap')
            pct_of_daily = (vol / total_daily * 100) if total_daily > 0 else 0

            gap_rows.append({
                'Corridor':         corridor,
                'Day_Type':         day_label,
                'Time_Period':      period,
                'Volume':           round(vol),
                'Pct_of_Daily':     round(pct_of_daily, 1),
                'Transit_Coverage': coverage,
                'Flag':             'GAP'  if coverage == 'Outside Service Hours' and pct_of_daily >= 5 else
                                    'Note' if coverage == 'Outside Service Hours' and 0 < pct_of_daily < 5 else ''
            })

In [ ]:
gap_df  = pd.DataFrame(gap_rows)
flagged = gap_df[gap_df['Flag'] == 'GAP'].sort_values(
    ['Corridor', 'Day_Type', 'Volume'], ascending=[True, True, False])

In [ ]:
gap_df.to_csv( os.path.join(OUTPUT_DIR, 'gap_inventory', 'gap_inventory_full.csv'),    index=False)
flagged.to_csv(os.path.join(OUTPUT_DIR, 'gap_inventory', 'gap_inventory_flagged.csv'), index=False)

In [ ]:
print(f"  Records evaluated: {len(gap_df):,}")
print(f"  Flagged gaps (≥5% of daily, outside service hours): {len(flagged)}")
print(f"  Minor pockets (<5%): {len(gap_df[gap_df['Flag'] == 'Note'])}\n")

In [ ]:
if not flagged.empty:
    for corridor in present_corridors:
        c_flag = flagged[flagged['Corridor'] == corridor]
        if not c_flag.empty:
            print(f"  {corridor}:")
            for _, row in c_flag.iterrows():
                print(f"    {row['Day_Type']} | {row['Time_Period']}: "
                      f"{row['Volume']:,} trips ({row['Pct_of_Daily']}% of daily)")

In [ ]:
# Gap summary heatmap
gap_pivot_rows = []
for corridor in present_corridors:
    row = {'Corridor': corridor}
    for day_label in [v for k, v in DAY_LABELS.items() if k != '0: All Days (M-Su)']:
        subset = gap_df[
            (gap_df['Corridor'] == corridor) &
            (gap_df['Day_Type'] == day_label) &
            (gap_df['Transit_Coverage'] == 'Outside Service Hours')
        ]
        row[day_label] = round(subset['Pct_of_Daily'].sum(), 1)
    gap_pivot_rows.append(row)

In [ ]:
gap_pivot = pd.DataFrame(gap_pivot_rows).set_index('Corridor')

In [ ]:
make_heatmap(
    pivot=gap_pivot,
    title=('Arrivals Outside KT Service Hours — % of Daily Total\n'
           'Higher = more vehicle arrivals occur when no KT service operates'),
    cbar_label='% of Daily Arrivals Outside Service Hours',
    cmap='Oranges', vmin=0, vmax=50, fmt='.1f',
    outpath=os.path.join(OUTPUT_DIR, 'gap_inventory', 'gap_summary_heatmap.pdf'),
    figsize=(12, 6)
)

=============================================================================
ANALYSIS 6: EQUITY & HOUSEHOLD ANALYSIS
=============================================================================

In [ ]:
print("\n" + "=" * 70)
print("ANALYSIS 6: EQUITY AND HOUSEHOLD CHARACTERISTICS")
print("=" * 70)
print("  Proportions from StreetLight reflect origin-area demographics,")
print("  not directly observed traveler characteristics.\n")

In [ ]:
os.makedirs(os.path.join(OUTPUT_DIR, 'equity_analysis'), exist_ok=True)

In [ ]:
if df_equity is not None:

    eq_hourly = df_equity[df_equity['Hour'].notna()].copy()

    # --- 6a: Disability share heatmap per corridor ---
    print("  6a: Disability share heatmaps...")
    for corridor in present_corridors:
        cdata = eq_hourly[eq_hourly[EQ_CORRIDOR_COL] == corridor]
        if cdata.empty:
            continue

        piv = cdata.pivot_table(
            values=['With a disability_trips', EQ_VOL_COL],
            index='Period', columns='Day_Label', aggfunc='sum'
        )
        if piv.empty:
            continue

        vol_piv  = piv[EQ_VOL_COL]
        dis_piv  = piv['With a disability_trips']
        pct_piv  = (dis_piv / vol_piv * 100).fillna(0)
        pct_piv  = pct_piv.reindex([p for p in PERIOD_ORDER if p in pct_piv.index])
        pct_piv  = pct_piv[[c for c in day_col_order if c in pct_piv.columns]]

        make_heatmap(
            pivot=pct_piv,
            title=(f'Share of Trips — Travelers with Disabilities\n{corridor}\n'
                   f'Green dashed outline = KT service hours'),
            cbar_label='% of Arriving Trips (with disability)',
            cmap='YlOrRd', vmin=0, vmax=pct_piv.values.max(), fmt='.1f',
            outpath=os.path.join(OUTPUT_DIR, 'equity_analysis', f'Disability_{corridor}.pdf'),
            transit_cols=TRANSIT_COL_MAP, transit_hours=TRANSIT_HOURS
        )
        print(f"    {corridor}: disability heatmap saved")

    # --- 6b: Cross-corridor disability summary bar chart ---
    print("\n  6b: Cross-corridor disability summary...")
    disability_summary = []
    for corridor in present_corridors:
        cdata = df_equity[df_equity[EQ_CORRIDOR_COL] == corridor]
        if cdata.empty:
            continue
        total_trips = cdata[EQ_VOL_COL].sum()
        dis_trips   = cdata['With a disability_trips'].sum()
        disability_summary.append({
            'Corridor':     corridor,
            'Pct_Disability': (dis_trips / total_trips * 100) if total_trips > 0 else 0
        })

    dis_df = pd.DataFrame(disability_summary).sort_values('Pct_Disability', ascending=True)
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(dis_df['Corridor'], dis_df['Pct_Disability'], color='#e67e22')
    for bar, val in zip(bars, dis_df['Pct_Disability']):
        ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
                f'{val:.1f}%', va='center', fontsize=9)
    ax.set_xlabel('% of Arriving Trips')
    ax.set_title('Share of Trips by Travelers with Disabilities — by Corridor\n'
                 '(Based on origin-area demographics)', fontsize=11)
    ax.set_xlim(0, dis_df['Pct_Disability'].max() * 1.2)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'equity_analysis', 'disability_by_corridor.pdf'),
                dpi=300, bbox_inches='tight')
    plt.close()

    # --- 6c: Outside-service disability focus ---
    print("  6c: Disability share outside service hours...")
    out_eq = eq_hourly[eq_hourly['Within_Transit'] == False]
    dis_out_summary = []
    for corridor in present_corridors:
        cdata = out_eq[out_eq[EQ_CORRIDOR_COL] == corridor]
        if cdata.empty:
            continue
        total = cdata[EQ_VOL_COL].sum()
        dis   = cdata['With a disability_trips'].sum()
        dis_out_summary.append({
            'Corridor':              corridor,
            'Outside_Svc_Volume':    round(total),
            'Disability_Trips_OutSvc': round(dis),
            'Pct_Disability_OutSvc': round((dis / total * 100) if total > 0 else 0, 1)
        })
    pd.DataFrame(dis_out_summary).to_csv(
        os.path.join(OUTPUT_DIR, 'equity_analysis', 'disability_outside_service.csv'), index=False)
    print("    Saved disability_outside_service.csv")

In [ ]:
if df_household is not None:

    hh_hourly = df_household[df_household['Hour'].notna()].copy()

    # --- 6d: Zero-vehicle heatmap per corridor ---
    print("\n  6d: Zero-vehicle household heatmaps...")
    for corridor in present_corridors:
        cdata = hh_hourly[hh_hourly[EQ_CORRIDOR_COL] == corridor]
        if cdata.empty:
            continue

        piv = cdata.pivot_table(
            values=['No vehicle available_trips', EQ_VOL_COL],
            index='Period', columns='Day_Label', aggfunc='sum'
        )
        if piv.empty:
            continue

        vol_piv = piv[EQ_VOL_COL]
        zv_piv  = piv['No vehicle available_trips']
        pct_piv = (zv_piv / vol_piv * 100).fillna(0)
        pct_piv = pct_piv.reindex([p for p in PERIOD_ORDER if p in pct_piv.index])
        pct_piv = pct_piv[[c for c in day_col_order if c in pct_piv.columns]]

        make_heatmap(
            pivot=pct_piv,
            title=(f'Share of Trips — Zero-Vehicle Households\n{corridor}\n'
                   f'Green dashed outline = KT service hours'),
            cbar_label='% of Arriving Trips (zero-vehicle households)',
            cmap='YlOrRd', vmin=0, vmax=pct_piv.values.max(), fmt='.1f',
            outpath=os.path.join(OUTPUT_DIR, 'equity_analysis', f'ZeroVehicle_{corridor}.pdf'),
            transit_cols=TRANSIT_COL_MAP, transit_hours=TRANSIT_HOURS
        )
        print(f"    {corridor}: zero-vehicle heatmap saved")

    # --- 6e: Cross-corridor zero-vehicle summary ---
    print("\n  6e: Cross-corridor zero-vehicle summary...")
    zv_summary = []
    for corridor in present_corridors:
        cdata = df_household[df_household[EQ_CORRIDOR_COL] == corridor]
        if cdata.empty:
            continue
        total = cdata[EQ_VOL_COL].sum()
        zv    = cdata['No vehicle available_trips'].sum()
        zv_summary.append({
            'Corridor':      corridor,
            'Pct_Zero_Veh':  (zv / total * 100) if total > 0 else 0
        })

    zv_df = pd.DataFrame(zv_summary).sort_values('Pct_Zero_Veh', ascending=True)
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(zv_df['Corridor'], zv_df['Pct_Zero_Veh'], color='#2980b9')
    for bar, val in zip(bars, zv_df['Pct_Zero_Veh']):
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
                f'{val:.1f}%', va='center', fontsize=9)
    ax.set_xlabel('% of Arriving Trips')
    ax.set_title('Share of Trips from Zero-Vehicle Households — by Corridor\n'
                 '(Based on origin-area demographics)', fontsize=11)
    ax.set_xlim(0, zv_df['Pct_Zero_Veh'].max() * 1.2)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'equity_analysis', 'zero_vehicle_by_corridor.pdf'),
                dpi=300, bbox_inches='tight')
    plt.close()

    # --- 6f: Combined equity summary table ---
    print("\n  6f: Combined equity summary table...")
    equity_rows = []
    for corridor in present_corridors:
        eq_c  = df_equity[df_equity[EQ_CORRIDOR_COL] == corridor]   if df_equity    is not None else pd.DataFrame()
        hh_c  = df_household[df_household[EQ_CORRIDOR_COL] == corridor] if df_household is not None else pd.DataFrame()
        row   = {'Corridor': corridor}
        if not eq_c.empty:
            t = eq_c[EQ_VOL_COL].sum()
            row['Pct_Disability'] = round(eq_c['With a disability_trips'].sum() / t * 100, 1) if t > 0 else 0
            row['Pct_Hispanic']   = round(eq_c['Hispanic_trips'].sum()          / t * 100, 1) if t > 0 else 0
            row['Pct_Foreign_Born'] = round(eq_c['Foreign Born_trips'].sum()    / t * 100, 1) if t > 0 else 0
        if not hh_c.empty:
            t = hh_c[EQ_VOL_COL].sum()
            row['Pct_Zero_Veh']   = round(hh_c['No vehicle available_trips'].sum() / t * 100, 1) if t > 0 else 0
            row['Pct_Renter']     = round(hh_c['Renter occupied_trips'].sum()       / t * 100, 1) if t > 0 else 0
        equity_rows.append(row)

    equity_summary_df = pd.DataFrame(equity_rows)
    equity_summary_df.to_csv(
        os.path.join(OUTPUT_DIR, 'equity_analysis', 'equity_summary_all_corridors.csv'), index=False)
    print("\n  Equity Summary:")
    print(equity_summary_df.to_string(index=False))

=============================================================================
ANALYSIS 7: TRIP LENGTH vs TRAVEL TIME — SCATTER BUBBLE CHART
=============================================================================

In [ ]:
print("\n" + "=" * 70)
print("ANALYSIS 7: TRIP LENGTH vs TRAVEL TIME — CORRIDOR COMPARISON")
print("=" * 70)

In [ ]:
bubble_rows = []

In [ ]:
for corridor in present_corridors:
    cdata = df[df[CORRIDOR_COL] == corridor]
    if cdata.empty:
        continue

    avg_length   = weighted_avg(cdata['Avg All Trip Length (mi)'], cdata[VOLUME_COL])
    avg_time_min = weighted_avg(cdata['Avg All Travel Time (sec)'], cdata[VOLUME_COL]) / 60
    total_vol    = cdata[VOLUME_COL].sum()
    typology     = typology_df[typology_df['Corridor'] == corridor]['Typology'].values
    typology     = typology[0] if len(typology) > 0 else 'Unknown'

    bubble_rows.append({
        'Corridor':    corridor,
        'Avg_Length':  avg_length,
        'Avg_Time':    avg_time_min,
        'Total_Vol':   total_vol,
        'Typology':    typology
    })

In [ ]:
bubble_df = pd.DataFrame(bubble_rows)

In [ ]:
typology_colors = {
    'Neighborhood-Serving': '#2ecc71',
    'Regional Draw':        '#e74c3c',
    'Mixed':                '#f39c12'
}

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

In [ ]:
for _, row in bubble_df.iterrows():
    color  = typology_colors.get(row['Typology'], '#95a5a6')
    size   = (row['Total_Vol'] / bubble_df['Total_Vol'].max()) * 3000
    ax.scatter(row['Avg_Length'], row['Avg_Time'],
               s=size, color=color, alpha=0.7, edgecolors='white', linewidth=1.5)
    ax.annotate(row['Corridor'],
                xy=(row['Avg_Length'], row['Avg_Time']),
                xytext=(8, 4), textcoords='offset points',
                fontsize=9, fontweight='bold')

In [ ]:
# Legend for typology
for typology, color in typology_colors.items():
    ax.scatter([], [], s=150, color=color, alpha=0.7, label=typology)

In [ ]:
# Legend for bubble size
for vol_pct, label in [(0.25, '25% of max volume'), (0.5, '50%'), (1.0, '100%')]:
    ax.scatter([], [], s=vol_pct * 3000, color='#7f8c8d', alpha=0.5, label=label)

In [ ]:
ax.set_xlabel('Avg All Trip Length (miles)', fontsize=11)
ax.set_ylabel('Avg All Travel Time (minutes)', fontsize=11)
ax.set_title(
    'Trip Length vs. Travel Time by Commercial Node\n'
    'Bubble size = total arriving volume   Color = corridor typology\n'
    'Nodes above the trend line have longer travel times relative to distance — '
    'possible network inefficiency',
    fontsize=11
)

In [ ]:
# Reference line — average time-per-mile ratio across all corridors
ratio = bubble_df['Avg_Time'].sum() / bubble_df['Avg_Length'].sum()
x_range = np.linspace(bubble_df['Avg_Length'].min() * 0.8,
                       bubble_df['Avg_Length'].max() * 1.1, 100)
ax.plot(x_range, x_range * ratio,
        color='#bdc3c7', linestyle='--', linewidth=1, label='Average time/distance trend')

In [ ]:
ax.legend(loc='upper left', fontsize=8, framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'summary_tables', 'length_time_bubble.pdf'),
            dpi=300, bbox_inches='tight')
plt.close()
print("  Bubble chart saved.")

=============================================================================
ANALYSIS 8: TRIP LENGTH & TRAVEL TIME BY HOUR — SMALL MULTIPLES
=============================================================================

In [ ]:
print("\n" + "=" * 70)
print("ANALYSIS 8: TRIP LENGTH & TRAVEL TIME BY HOUR — SMALL MULTIPLES")
print("=" * 70)

In [ ]:
nrows = int(np.ceil(len(present_corridors) / 2))
fig, axes = plt.subplots(nrows=nrows, ncols=2,
                          figsize=(16, 4 * nrows), sharex=False)
axes = axes.flatten()

In [ ]:
for idx, corridor in enumerate(present_corridors):
    cdata = df_hourly[df_hourly[CORRIDOR_COL] == corridor].copy()
    if cdata.empty:
        continue

    hourly = cdata.groupby('Hour').agg(
        avg_length = ('Avg All Trip Length (mi)',   'mean'),
        avg_time   = ('Avg All Travel Time (sec)',  'mean'),
        volume     = (VOLUME_COL,                   'sum')
    ).reset_index()
    hourly['avg_time_min'] = hourly['avg_time'] / 60

    # Normalize both to 0–1 so they plot on same axis
    def norm(s):
        rng = s.max() - s.min()
        return (s - s.min()) / rng if rng > 0 else s * 0

    hourly['length_norm'] = norm(hourly['avg_length'])
    hourly['time_norm']   = norm(hourly['avg_time_min'])

    ax = axes[idx]
    ax.plot(hourly['Hour'], hourly['length_norm'],
            color='#2980b9', linewidth=2, marker='o', markersize=3, label='Trip Length')
    ax.plot(hourly['Hour'], hourly['time_norm'],
            color='#e74c3c', linewidth=2, marker='o', markersize=3, label='Travel Time')

    # Shade transit service hours (use Sunday as most restrictive — adjust if needed)
    svc = TRANSIT_HOURS.get('sunday', {})
    if svc:
        ax.axvspan(svc['start'], svc['end'], alpha=0.08, color='green', label='KT service hrs')

    # Mark divergence — where time rises faster than length
    hourly['divergence'] = hourly['time_norm'] - hourly['length_norm']
    peak_div_hour = hourly.loc[hourly['divergence'].idxmax(), 'Hour']
    ax.axvline(peak_div_hour, color='orange', linestyle=':', linewidth=1.5,
               label=f'Peak divergence ({int(peak_div_hour)}:00)')

    ax.set_title(f'{corridor}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Hour of Day', fontsize=8)
    ax.set_ylabel('Normalized Value', fontsize=8)
    ax.set_xticks([0, 4, 8, 12, 16, 20, 23])
    ax.set_xticklabels(['12a', '4a', '8a', '12p', '4p', '8p', '11p'], fontsize=7)
    ax.set_ylim(-0.05, 1.15)
    ax.grid(True, alpha=0.3)

    # Add raw value annotations at peak hours
    peak_hour = hourly.loc[hourly['volume'].idxmax(), 'Hour']
    peak_row  = hourly[hourly['Hour'] == peak_hour].iloc[0]
    ax.annotate(
        f"{peak_row['avg_length']:.1f}mi\n{peak_row['avg_time_min']:.0f}min",
        xy=(peak_hour, 1.0),
        fontsize=7, color='#2c3e50',
        ha='center', va='bottom',
        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7)
    )

    if idx == 0:
        ax.legend(fontsize=7, loc='upper right')

In [ ]:
for idx in range(len(present_corridors), len(axes)):
    axes[idx].set_visible(False)

In [ ]:
fig.suptitle(
    'Trip Length vs. Travel Time by Hour — All Commercial Corridors\n'
    'Blue = trip length (miles)   Red = travel time (minutes)   Both normalized to 0–1\n'
    'Divergence = travel time rising faster than distance (congestion or indirect routing)\n'
    'Green shading = KT Sunday service hours (most restrictive day)',
    fontsize=11, y=1.01
)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'summary_tables', 'length_time_small_multiples.pdf'),
            dpi=300, bbox_inches='tight')
plt.close()
print("  Small multiples chart saved.")

=============================================================================
EXPORT SUMMARY
=============================================================================

In [ ]:
print("\n" + "=" * 70)
print("EXPORTING SUMMARY FILES")
print("=" * 70)

In [ ]:
excel_path = os.path.join(OUTPUT_DIR, 'summary_tables', 'BI_Corridor_Analysis_Summary.xlsx')
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    typology_df.to_excel(writer, sheet_name='Corridor_Typology',    index=False)
    length_df.to_excel(  writer, sheet_name='Origin_Distance',      index=False)
    flagged.to_excel(    writer, sheet_name='Flagged_Gaps',          index=False)
    gap_df.to_excel(     writer, sheet_name='Gap_Inventory_Full',    index=False)
    gap_pivot.to_excel(  writer, sheet_name='Gap_Pct_By_DayType')
    if 'equity_summary_df' in dir():
        equity_summary_df.to_excel(writer, sheet_name='Equity_Summary', index=False)

In [ ]:
print(f"  Saved: {excel_path}")

=============================================================================
FINDINGS 
=============================================================================

In [ ]:
print("\n" + "=" * 70)
print("FINDINGS")
print("=" * 70)
print(f"Date:        {datetime.now().strftime('%Y-%m-%d')}")
print(f"Data source: StreetLight Destination, Summer 2024, All Vehicles CVD+")
print(f"Zone role:   Commercial nodes as DESTINATIONS")
print(f"Corridors:   {', '.join(present_corridors)}")
if missing_corridors:
    print(f"Missing:     {', '.join(missing_corridors)}")

In [ ]:
print("\nCORRIDOR TYPOLOGY:")
for _, row in typology_df.iterrows():
    print(f"  {row['Corridor']}: {row['Typology']} — "
          f"{row['Pct_Nearby_Trips']}% nearby-origin, "
          f"{row['Pct_Distant_Trips']}% distant-origin, "
          f"avg origin distance {row['Avg_Origin_Distance_mi']} mi")

In [ ]:
print("\nSERVICE HOUR GAPS (≥5% of daily arrivals, outside KT service hours):")
if flagged.empty:
    print("  No significant gaps at the ≥5% threshold.")
else:
    print(f"  {len(flagged)} gap periods across {flagged['Corridor'].nunique()} corridors.")
    print("  See gap_inventory_flagged.csv. These are volume observations,")
    print("  not confirmed unmet demand.")

In [ ]:
print("\nDATA LIMITATIONS:")
print("  - StreetLight volumes are ML estimates, not physical counts.")
print("  - Demographic proportions reflect origin-area census data,")
print("    not directly observed traveler characteristics.")
print("  - LYNWOODCENTER" + (
    " is present in data." if "LYNWOODCENTER" in present_corridors
    else " not yet in data — add zone and rerun to include south island node."))

In [ ]:
print(f"\nAnalysis complete: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 70)